In [1]:
from pprint import pprint
from IPython.display import display

from fda_ode import *

Орегонатор (Oregonator) — это математическая модель, предложенная Ричардом Филдом и Ричардом Нойесом в 1974 году в Орегонском университете. Она представляет собой упрощенную версию реального химического механизма реакции Белоусова–Жаботинского (БЖ) — классического примера автоколебательной химической системы.
Основная ценность Орегонатора для вычислительной математики заключается в том, что эта задача является экстремально жесткой. Она служит эталонным тестом для проверки численных методов, таких как неявные методы Рунге-Кутты (включая рассмотренный ранее Radau IIA).

В безразмерном виде классический Орегонатор описывается системой трех обыкновенных дифференциальных уравнений:
$$\begin{cases} \frac{dx}{dt} = \frac{1}{\varepsilon} \left( x + y - x y - q x^2 \right) \\ \frac{dy}{dt} = - y + z - x y \\ \frac{dz}{dt} = \delta (x - z) \end{cases}$$ 
## Переменные и их физический смысл:

* $x$ — безразмерная концентрация бромноватистой кислоты ($\text{HBrO}_2$). Это ключевой промежуточный продукт, который обеспечивает автокатализ (лавинообразное ускорение реакции).
* $y$ — безразмерная концентрация бромид-ионов ($\text{Br}^-$). Выступает в роли «ингибитора». Когда бромида много, он подавляет автокатализ.
* $z$ — безразмерная концентрация окисленной формы катализатора ($\text{Ce}^{4+}$). Именно изменение этой концентрации окрашивает раствор в разные цвета.

## Параметры жесткости системы:
Жесткость системы задается малыми параметрами $\varepsilon$ и $q$. Типичные «тестовые» значения, принятые в численных методах:

* $\varepsilon = 0.01$ (или даже $10^{-4}$ для экстремальных тестов) — регулирует колоссальную разницу в скоростях реакций.
* $\delta = 1.0$ — определяет скорость восстановления катализатора.
* $q = 2 \cdot 10^{-4}$ — малый стехиометрический параметр.

In [2]:
eps, q, delta, t, h = symbols(r'varepsilon, q, delta, t, h', real=True)
x, y, z = symbols('x, y, z', cls=Function)

In [3]:
eq1 = x(t).diff(t) - 1/eps*(x(t) + y(t) - x(t)*y(t) - q*x(t)**2)
eq1

Derivative(x(t), t) - (-q*x(t)**2 - x(t)*y(t) + x(t) + y(t))/varepsilon

In [4]:
eq2 = y(t).diff(t) - (-y(t) + z(t) - x(t)*y(t))
eq2

x(t)*y(t) + y(t) - z(t) + Derivative(y(t), t)

In [5]:
eq3 = z(t).diff(t) - delta*(x(t) - z(t))
eq3

-delta*(x(t) - z(t)) + Derivative(z(t), t)

In [6]:
init((x, y, z), t, h)

In [7]:
def f(y):
    return Matrix([\
        1/eps*(y[0] + y[1] - y[0]*y[1] - q*y[0]**2),\
        -y[1] + y[2] - y[0]*y[1],\
        delta*(y[0] - y[2]),\
    ])

s = (('x(t)', 'y[0]'), ('y(t)', 'y[1]'), ('z(t)', 'y[2]'),\
     (r'varepsilon', 'eps'), (r'delta', 'dlt'))
def replace(r):
    for a, b in s:
        r = r.replace(a, b)
    return r

## RadauIA 1 order

In [8]:
%%time
F = RadauIA1(f)
for r in F:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


(2*q*x*x_{t} + x*y_{t} + x_{t}*y - x_{t} - y_{t})/varepsilon

x_{t} + \frac{q x^{2} + x y - x - y}{\varepsilon}
+h\left(
\frac{2 q x x_{t} + x y_{t} + x_{t} y - x_{t} - y_{t}}{\varepsilon}
\right)

h^0 =>


...

x y + y + y_{t} - z
+h\left(
x y_{t} + x_{t} y + y_{t} - z_{t}
\right)

h^0 =>


-delta*x + delta*z + z_{t}

h^1 =>


-delta*(x_{t} - z_{t})

- \delta x + \delta z + z_{t}
+h\left(
- \delta \left(x_{t} - z_{t}\right)
\right)

CPU times: user 220 ms, sys: 10.9 ms, total: 231 ms
Wall time: 231 ms


In [9]:
%%time
fda = FDA(F)
for r in fda:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


-(x - 1)*(x*y + y - z)/varepsilon - (2*q*x + y - 1)*(q*x**2 + x*y - x - y)/varepsilon**2

x_{t} + \frac{q x^{2} + x y - x - y}{\varepsilon}
+h\left(
- \frac{\left(x - 1\right) \left(x y + y - z\right)}{\varepsilon} - \frac{\left(2 q x + y - 1\right) \left(q x^{2} + x y - x - y\right)}{\varepsilon^{2}}
\right)

h^0 =>


...

x y + y + y_{t} - z
+h\left(
- \delta x + \delta z - x^{2} y - 2 x y + x z - y + z - \frac{y \left(q x^{2} + x y - x - y\right)}{\varepsilon}
\right)

h^0 =>


-delta*x + delta*z + z_{t}

h^1 =>


delta**2*(x - z) + delta*(q*x**2 + x*y - x - y)/varepsilon

- \delta x + \delta z + z_{t}
+h\left(
\delta^{2} \left(x - z\right) + \frac{\delta \left(q x^{2} + x y - x - y\right)}{\varepsilon}
\right)

CPU times: user 246 ms, sys: 4.12 ms, total: 250 ms
Wall time: 247 ms


In [11]:
print(replace(str(fda[0][1].collect(eps, factor))))

-(y[0] - 1)*(y[0]*y[1] + y[1] - y[2])/eps - (2*q*y[0] + y[1] - 1)*(q*y[0]**2 + y[0]*y[1] - y[0] - y[1])/eps**2


## Gauss–Legendre

In [12]:
%%time
F = GaussLegendre2(f)
for r in F:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


x_{tt}/2 + (2*q*x*x_{t} + x*y_{t} + x_{t}*y - x_{t} - y_{t})/(2*varepsilon)

h^2 =>


...

h^1 =>


-(delta*x_{t} - delta*z_{t} - z_{tt})/2

h^2 =>


-delta*(x_{tt} - z_{tt})/4

- \delta x + \delta z + z_{t}
+h\left(
- \frac{\delta x_{t} - \delta z_{t} - z_{tt}}{2}
\right)
+h^2\left(
- \frac{\delta \left(x_{tt} - z_{tt}\right)}{4}
\right)

CPU times: user 259 ms, sys: 0 ns, total: 259 ms
Wall time: 256 ms


In [13]:
%%time
fda = FDA(F)
for r in fda:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


0

h^2 =>


...

h^1 =>


0

h^2 =>


-delta**3*(x - z)/4 - delta*(delta*q*x**2 + delta*x*y - delta*x - delta*y + x**2*y - x*z - y + z)/(4*varepsilon) - delta*(2*q*x + y - 1)*(q*x**2 + x*y - x - y)/(4*varepsilon**2)

- \delta x + \delta z + z_{t}
+h\left(
0
\right)
+h^2\left(
- \frac{\delta^{3} \left(x - z\right)}{4} - \frac{\delta \left(\delta q x^{2} + \delta x y - \delta x - \delta y + x^{2} y - x z - y + z\right)}{4 \varepsilon} - \frac{\delta \left(2 q x + y - 1\right) \left(q x^{2} + x y - x - y\right)}{4 \varepsilon^{2}}
\right)

CPU times: user 847 ms, sys: 3.73 ms, total: 850 ms
Wall time: 849 ms


In [14]:
print(replace(str(fda[0][2].collect(eps, factor))))

(y[0] - 1)*(dlt*y[0] - dlt*y[2] + y[0]**2*y[1] + 2*y[0]*y[1] - y[0]*y[2] + y[1] - y[2])/(4*eps) + (4*q*y[0]**3*y[1] - 3*q*y[0]**2*y[2] - 2*q*y[0]*y[1] + 2*q*y[0]*y[2] + 3*y[0]**2*y[1]**2 - 3*y[0]**2*y[1] - 2*y[0]*y[1]**2 - 2*y[0]*y[1]*y[2] + 2*y[0]*y[2] - y[1]**2 + 2*y[1]*y[2] + y[1] - y[2])/(4*eps**2) + (q*y[0]**2 + y[0]*y[1] - y[0] - y[1])*(5*q**2*y[0]**2 + 5*q*y[0]*y[1] - 5*q*y[0] - q*y[1] + y[1]**2 - 2*y[1] + 1)/(4*eps**3)


## Crank–Nicolson

In [15]:
%%time
F = CrankNicolson2(f)
for r in F:
    r.prn(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


x_{tt}/2 + (2*q*x*x_{t} + x*y_{t} + x_{t}*y - x_{t} - y_{t})/(2*varepsilon)

h^2 =>


...

h^1 =>


-(delta*x_{t} - delta*z_{t} - z_{tt})/2

h^2 =>


-delta*(x_{tt} - z_{tt})/4


CPU times: user 179 ms, sys: 3.87 ms, total: 183 ms
Wall time: 179 ms


In [16]:
%%time
fda = FDA(F)
for r in fda:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


0

h^2 =>


...

h^1 =>


0

h^2 =>


-delta**3*(x - z)/4 - delta*(delta*q*x**2 + delta*x*y - delta*x - delta*y + x**2*y - x*z - y + z)/(4*varepsilon) - delta*(2*q*x + y - 1)*(q*x**2 + x*y - x - y)/(4*varepsilon**2)

- \delta x + \delta z + z_{t}
+h\left(
0
\right)
+h^2\left(
- \frac{\delta^{3} \left(x - z\right)}{4} - \frac{\delta \left(\delta q x^{2} + \delta x y - \delta x - \delta y + x^{2} y - x z - y + z\right)}{4 \varepsilon} - \frac{\delta \left(2 q x + y - 1\right) \left(q x^{2} + x y - x - y\right)}{4 \varepsilon^{2}}
\right)

CPU times: user 758 ms, sys: 7.91 ms, total: 766 ms
Wall time: 761 ms


In [17]:
print(replace(str(fda[0][2].collect(eps, factor))))

(y[0] - 1)*(dlt*y[0] - dlt*y[2] + y[0]**2*y[1] + 2*y[0]*y[1] - y[0]*y[2] + y[1] - y[2])/(4*eps) + (5*q*y[0]**3*y[1] + q*y[0]**2*y[1] - 4*q*y[0]**2*y[2] - 2*q*y[0]*y[1] + 2*q*y[0]*y[2] + 4*y[0]**2*y[1]**2 - 4*y[0]**2*y[1] - 2*y[0]*y[1]**2 - 3*y[0]*y[1]*y[2] - y[0]*y[1] + 3*y[0]*y[2] - 2*y[1]**2 + 3*y[1]*y[2] + y[1] - y[2])/(4*eps**2) + (q*y[0]**2 + y[0]*y[1] - y[0] - y[1])*(6*q**2*y[0]**2 + 6*q*y[0]*y[1] - 6*q*y[0] - 2*q*y[1] + y[1]**2 - 2*y[1] + 1)/(4*eps**3)


## RadauIIA 3 order

In [18]:
%%time
F = RadauIIA3(f)
for r in F:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


x_{tt}/2 + (8*q*x*x_{t} + x**2*y + 4*x*y_{t} - x*z + 4*x_{t}*y - 4*x_{t} - y - 4*y_{t} + z)/(6*varepsilon) + (2*q*x + y - 1)*(q*x**2 + x*y - x - y)/(6*varepsilon**2)

h^2 =>


...

h^2 =>


-delta*(2*q*x*x_{t} + x*y_{t} + x_{t}*y - x_{t} - y_{t})/(6*varepsilon) - (delta**2*x_{t} - delta**2*z_{t} + 2*delta*x_{tt} - 2*delta*z_{tt} - z_{ttt})/6

h^3 =>


-delta*(3*delta*x_{tt} - 3*delta*z_{tt} + 4*x_{ttt} - 4*z_{ttt})/36 - delta*(2*q*x*x_{tt} + 2*q*x_{t}**2 + x*y_{tt} + x_{tt}*y - x_{tt} + 2*x_{t}*y_{t} - y_{tt})/(12*varepsilon)

- \delta x + \delta z + z_{t}
+h\left(
- \frac{\delta \left(q x^{2} + x y - x - y\right)}{6 \varepsilon} - \frac{\delta^{2} x - \delta^{2} z + 4 \delta x_{t} - 4 \delta z_{t} - 3 z_{tt}}{6}
\right)
+h^2\left(
- \frac{\delta \left(2 q x x_{t} + x y_{t} + x_{t} y - x_{t} - y_{t}\right)}{6 \varepsilon} - \frac{\delta^{2} x_{t} - \delta^{2} z_{t} + 2 \delta x_{tt} - 2 \delta z_{tt} - z_{ttt}}{6}
\right)
+h^3\left(
- \frac{\delta \left(3 \delta x_{tt} - 3 \delta z_{tt} + 4 x_{ttt} - 4 z_{ttt}\right)}{36} - \frac{\delta \left(2 q x x_{tt} + 2 q x_{t}^{2} + x y_{tt} + x_{tt} y - x_{tt} + 2 x_{t} y_{t} - y_{tt}\right)}{12 \varepsilon}
\right)

CPU times: user 3.8 s, sys: 23.8 ms, total: 3.82 s
Wall time: 3.81 s


In [19]:
%%time
fda = FDA(F)
for r in fda:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


0

h^2 =>


...

h^2 =>


0

h^3 =>


delta**4*(x - z)/36 + delta*(delta**2*q*x**2 + delta**2*x*y - delta**2*x - delta**2*y + delta*x**2*y + delta*x**2 - 2*delta*x*z - delta*x - delta*y + 2*delta*z + x**3*y + x**2*y - x**2*z - x*y - y + z)/(36*varepsilon) + delta*(2*delta*q**2*x**3 + 3*delta*q*x**2*y - 3*delta*q*x**2 - 2*delta*q*x*y + delta*x*y**2 - 2*delta*x*y + delta*x - delta*y**2 + delta*y + 5*q*x**3*y + q*x**2*y - 4*q*x**2*z - 2*q*x*y + 2*q*x*z + 4*x**2*y**2 - 4*x**2*y - 2*x*y**2 - 3*x*y*z - x*y + 3*x*z - 2*y**2 + 3*y*z + y - z)/(36*varepsilon**2) + delta*(q*x**2 + x*y - x - y)*(6*q**2*x**2 + 6*q*x*y - 6*q*x - 2*q*y + y**2 - 2*y + 1)/(36*varepsilon**3)

- \delta x + \delta z + z_{t}
+h\left(
0
\right)
+h^2\left(
0
\right)
+h^3\left(
\frac{\delta^{4} \left(x - z\right)}{36} + \frac{\delta \left(\delta^{2} q x^{2} + \delta^{2} x y - \delta^{2} x - \delta^{2} y + \delta x^{2} y + \delta x^{2} - 2 \delta x z - \delta x - \delta y + 2 \delta z + x^{3} y + x^{2} y - x^{2} z - x y - y + z\right)}{36 \varepsilon} + \frac{\delta \left(2 \delta q^{2} x^{3} + 3 \delta q x^{2} y - 3 \delta q x^{2} - 2 \delta q x y + \delta x y^{2} - 2 \delta x y + \delta x - \delta y^{2} + \delta y + 5 q x^{3} y + q x^{2} y - 4 q x^{2} z - 2 q x y + 2 q x z + 4 x^{2} y^{2} - 4 x^{2} y - 2 x y^{2} - 3 x y z - x y + 3 x z - 2 y^{2} + 3 y z + y - z\right)}{36 \varepsilon^{2}} + \frac{\delta \left(q x^{2} + x y - x - y\right) \left(6 q^{2} x^{2} + 6 q x y - 6 q x - 2 q y + y^{2} - 2 y + 1\right)}{36 \varepsilon^{3}}
\right)

CPU times: user 13.9 s, sys: 7.85 ms, total: 13.9 s
Wall time: 13.9 s


In [20]:
print(replace(str(fda[0][3].collect(eps, factor))))

-(y[0] - 1)*(dlt**2*y[0] - dlt**2*y[2] + dlt*y[0]**2 - dlt*y[0]*y[2] + dlt*y[0] - dlt*y[2] + y[0]**3*y[1] + 3*y[0]**2*y[1] - y[0]**2*y[2] + 3*y[0]*y[1] - 2*y[0]*y[2] + y[1] - y[2])/(36*eps) - (8*dlt*q*y[0]**3 - 7*dlt*q*y[0]**2*y[2] - 3*dlt*q*y[0]**2 + 2*dlt*q*y[0]*y[2] + 7*dlt*y[0]**2*y[1] - 7*dlt*y[0]**2 - 6*dlt*y[0]*y[1]*y[2] - 8*dlt*y[0]*y[1] + 6*dlt*y[0]*y[2] + 2*dlt*y[0] + 6*dlt*y[1]*y[2] + dlt*y[1] - dlt*y[2] + 10*q*y[0]**4*y[1] + 12*q*y[0]**3*y[1] - 9*q*y[0]**3*y[2] - 3*q*y[0]**2*y[2] - 2*q*y[0]*y[1] + 2*q*y[0]*y[2] + 15*y[0]**3*y[1]**2 - 9*y[0]**3*y[1] + 7*y[0]**2*y[1]**2 - 19*y[0]**2*y[1]*y[2] - 11*y[0]**2*y[1] + 8*y[0]**2*y[2] - 15*y[0]*y[1]**2 + 6*y[0]*y[1]*y[2] - y[0]*y[1] + 5*y[0]*y[2]**2 + 3*y[0]*y[2] - 7*y[1]**2 + 13*y[1]*y[2] + y[1] - 5*y[2]**2 - y[2])/(36*eps**2) - (37*q**2*y[0]**4*y[1] + 10*q**2*y[0]**3*y[1] - 28*q**2*y[0]**3*y[2] - 14*q**2*y[0]**2*y[1] + 14*q**2*y[0]**2*y[2] + 51*q*y[0]**3*y[1]**2 - 51*q*y[0]**3*y[1] - 23*q*y[0]**2*y[1]**2 - 35*q*y[0]**2*y[1]*y[2] - 

## "original" Runge–Kutta

In [21]:
%%time
F = RungeKutta4(f)
for r in F:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


x_{tt}/2 - (x - 1)*(x*y + y - z)/(2*varepsilon) - (2*q*x + y - 1)*(q*x**2 + x*y - x - y)/(2*varepsilon**2)

h^2 =>


...

h^3 =>


delta*(delta**2*q*x**2 + delta**2*x*y - delta**2*x - delta**2*y + delta*x**2*y + delta*x**2 - 2*delta*x*z - delta*x - delta*y + 2*delta*z + x**3*y + x**2*y - x**2*z - x*y - y + z)/(24*varepsilon) + delta*(2*delta*q**2*x**3 + 3*delta*q*x**2*y - 3*delta*q*x**2 - 2*delta*q*x*y + delta*x*y**2 - 2*delta*x*y + delta*x - delta*y**2 + delta*y + 5*q*x**3*y + q*x**2*y - 4*q*x**2*z - 2*q*x*y + 2*q*x*z + 4*x**2*y**2 - 4*x**2*y - 2*x*y**2 - 3*x*y*z - x*y + 3*x*z - 2*y**2 + 3*y*z + y - z)/(24*varepsilon**2) + delta*(q*x**2 + x*y - x - y)*(6*q**2*x**2 + 6*q*x*y - 6*q*x - 2*q*y + y**2 - 2*y + 1)/(24*varepsilon**3) + (delta**4*x - delta**4*z + z_{tttt})/24

h^4 =>


-delta*q*(2*q*x + y - 1)*(q*x**2 + x*y - x - y)**2/(16*varepsilon**4) - delta*(delta*q*x**3*y + delta*q*x**3 + delta*q*x**2*y - 2*delta*q*x**2*z + delta*x**2*y**2 - delta*x**2 - 2*delta*x*y*z - 2*delta*x*y + 2*delta*x*z - delta*y**2 + 2*delta*y*z + 2*q*x**4*y + 2*q*x**3*y - 2*q*x**3*z + 3*x**3*y**2 - 2*x**3*y + x**2*y**2 - 4*x**2*y*z - 2*x**2*y + 2*x**2*z - 3*x*y**2 + 2*x*y*z + x*z**2 - y**2 + 2*y*z - z**2)/(48*varepsilon**2) - delta*(q*x**2 + x*y - x - y)*(delta*q**2*x**2 + delta*q*x*y - delta*q*x - delta*q*y + 7*q*x**2*y + 4*q*x*y - 6*q*x*z - 2*q*y + 2*q*z + 3*x*y**2 - 3*x*y + y**2 - 2*y*z - 2*y + 2*z)/(48*varepsilon**3)

- \delta x + \delta z + z_{t}
+h\left(
\frac{\delta \left(q x^{2} + x y - x - y\right)}{2 \varepsilon} + \frac{\delta^{2} x - \delta^{2} z + z_{tt}}{2}
\right)
+h^2\left(
- \frac{\delta \left(\delta q x^{2} + \delta x y - \delta x - \delta y + x^{2} y - x z - y + z\right)}{6 \varepsilon} - \frac{\delta \left(2 q x + y - 1\right) \left(q x^{2} + x y - x - y\right)}{6 \varepsilon^{2}} - \frac{\delta^{3} x - \delta^{3} z - z_{ttt}}{6}
\right)
+h^3\left(
\frac{\delta \left(\delta^{2} q x^{2} + \delta^{2} x y - \delta^{2} x - \delta^{2} y + \delta x^{2} y + \delta x^{2} - 2 \delta x z - \delta x - \delta y + 2 \delta z + x^{3} y + x^{2} y - x^{2} z - x y - y + z\right)}{24 \varepsilon} + \frac{\delta \left(2 \delta q^{2} x^{3} + 3 \delta q x^{2} y - 3 \delta q x^{2} - 2 \delta q x y + \delta x y^{2} - 2 \delta x y + \delta x - \delta y^{2} + \delta y + 5 q x^{3} y + q x^{2} y - 4 q x^{2} z - 2 q x y + 2 q x z + 4 x^{2} y^{2} - 4 x^{2} y - 2 x y^{2} - 3 x y z - x y + 3 x z - 2 y^{2} + 3 y z 

In [22]:
%%time
fda = FDA(F)
for r in fda:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


0

h^2 =>


...

h^3 =>


0

h^4 =>


-delta*q*(2*q*x + y - 1)*(q*x**2 + x*y - x - y)**2/(16*varepsilon**4) - delta*(delta*q*x**3*y + delta*q*x**3 + delta*q*x**2*y - 2*delta*q*x**2*z + delta*x**2*y**2 - delta*x**2 - 2*delta*x*y*z - 2*delta*x*y + 2*delta*x*z - delta*y**2 + 2*delta*y*z + 2*q*x**4*y + 2*q*x**3*y - 2*q*x**3*z + 3*x**3*y**2 - 2*x**3*y + x**2*y**2 - 4*x**2*y*z - 2*x**2*y + 2*x**2*z - 3*x*y**2 + 2*x*y*z + x*z**2 - y**2 + 2*y*z - z**2)/(48*varepsilon**2) - delta*(q*x**2 + x*y - x - y)*(delta*q**2*x**2 + delta*q*x*y - delta*q*x - delta*q*y + 7*q*x**2*y + 4*q*x*y - 6*q*x*z - 2*q*y + 2*q*z + 3*x*y**2 - 3*x*y + y**2 - 2*y*z - 2*y + 2*z)/(48*varepsilon**3)

- \delta x + \delta z + z_{t}
+h\left(
0
\right)
+h^2\left(
0
\right)
+h^3\left(
0
\right)
+h^4\left(
- \frac{\delta q \left(2 q x + y - 1\right) \left(q x^{2} + x y - x - y\right)^{2}}{16 \varepsilon^{4}} - \frac{\delta \left(\delta q x^{3} y + \delta q x^{3} + \delta q x^{2} y - 2 \delta q x^{2} z + \delta x^{2} y^{2} - \delta x^{2} - 2 \delta x y z - 2 \delta x y + 2 \delta x z - \delta y^{2} + 2 \delta y z + 2 q x^{4} y + 2 q x^{3} y - 2 q x^{3} z + 3 x^{3} y^{2} - 2 x^{3} y + x^{2} y^{2} - 4 x^{2} y z - 2 x^{2} y + 2 x^{2} z - 3 x y^{2} + 2 x y z + x z^{2} - y^{2} + 2 y z - z^{2}\right)}{48 \varepsilon^{2}} - \frac{\delta \left(q x^{2} + x y - x - y\right) \left(\delta q^{2} x^{2} + \delta q x y - \delta q x - \delta q y + 7 q x^{2} y + 4 q x y - 6 q x z - 2 q y + 2 q z + 3 x y^{2} - 3 x y + y^{2} - 2 y z - 2 y + 2 z\right)}{48 \varepsilon^{3}}
\right)

CPU times: user 1min 46s, sys: 39.9 ms, total: 1min 46s
Wall time: 1min 46s


In [23]:
print(replace(str(fda[0][4].collect(eps, factor))))

q*(q*y[0]**2 + y[0]*y[1] - y[0] - y[1])**2*(23*q**2*y[0]**2 + 23*q*y[0]*y[1] - 23*q*y[0] - 3*q*y[1] + 5*y[1]**2 - 10*y[1] + 5)/(24*eps**5) + (2*dlt**2*q*y[0]**3 - 2*dlt**2*q*y[0]**2*y[2] + 2*dlt**2*y[0]**2*y[1] - 2*dlt**2*y[0]**2 - 2*dlt**2*y[0]*y[1]*y[2] - 2*dlt**2*y[0]*y[1] + 2*dlt**2*y[0]*y[2] + 2*dlt**2*y[1]*y[2] + 3*dlt*q*y[0]**4 - 3*dlt*q*y[0]**3*y[2] + dlt*q*y[0]**3 - dlt*q*y[0]**2*y[2] + 8*dlt*y[0]**3*y[1] - 3*dlt*y[0]**3 - 8*dlt*y[0]**2*y[1]*y[2] - 2*dlt*y[0]**2*y[1] - 2*dlt*y[0]**2*y[2] - dlt*y[0]**2 + 2*dlt*y[0]*y[1]*y[2] - 6*dlt*y[0]*y[1] + 5*dlt*y[0]*y[2]**2 + 6*dlt*y[0]*y[2] + 6*dlt*y[1]*y[2] - 5*dlt*y[2]**2 + 4*q*y[0]**5*y[1] + 8*q*y[0]**4*y[1] - 4*q*y[0]**4*y[2] + 4*q*y[0]**3*y[1] - 4*q*y[0]**3*y[2] + 10*y[0]**4*y[1]**2 - 4*y[0]**4*y[1] + 14*y[0]**3*y[1]**2 - 16*y[0]**3*y[1]*y[2] - 8*y[0]**3*y[1] + 4*y[0]**3*y[2] - 6*y[0]**2*y[1]**2 - 8*y[0]**2*y[1]*y[2] - 4*y[0]**2*y[1] + 6*y[0]**2*y[2]**2 + 4*y[0]**2*y[2] - 14*y[0]*y[1]**2 + 16*y[0]*y[1]*y[2] - 2*y[0]*y[2]**2 - 4*y[1]

## Dormand–Prince 5 order

In [24]:
%%time
F = DormandPrince5(f)
for r in F:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


x_{tt}/2 - (x - 1)*(x*y + y - z)/(2*varepsilon) - (2*q*x + y - 1)*(q*x**2 + x*y - x - y)/(2*varepsilon**2)

h^2 =>


...

h^4 =>


-delta*(delta**3*q*x**2 + delta**3*x*y - delta**3*x - delta**3*y + delta**2*x**2*y + 2*delta**2*x**2 - 3*delta**2*x*z - 2*delta**2*x - delta**2*y + 3*delta**2*z + delta*x**3*y + delta*x**3 + delta*x**2*y - 2*delta*x**2*z - delta*x*y - delta*x - delta*y + 2*delta*z + x**4*y + 2*x**3*y - x**3*z - x**2*z - 2*x*y + x*z - y + z)/(120*varepsilon) - delta*(2*delta**2*q**2*x**3 + 3*delta**2*q*x**2*y - 3*delta**2*q*x**2 - 2*delta**2*q*x*y + delta**2*x*y**2 - 2*delta**2*x*y + delta**2*x - delta**2*y**2 + delta**2*y + 5*delta*q*x**3*y + 6*delta*q*x**3 + delta*q*x**2*y - 9*delta*q*x**2*z - 3*delta*q*x**2 - 2*delta*q*x*y + 4*delta*q*x*z + 4*delta*x**2*y**2 + delta*x**2*y - 5*delta*x**2 - 2*delta*x*y**2 - 7*delta*x*y*z - 7*delta*x*y + 7*delta*x*z + 2*delta*x - 2*delta*y**2 + 7*delta*y*z + 2*delta*y - 2*delta*z + 8*q*x**4*y + 8*q*x**3*y - 7*q*x**3*z - 2*q*x**2*y - q*x**2*z - 2*q*x*y + 2*q*x*z + 11*x**3*y**2 - 7*x**3*y + 3*x**2*y**2 - 13*x**2*y*z - 7*x**2*y + 6*x**2*z - 11*x*y**2 + 6*x*y*z + x*y + 3*x

h^5 =>


delta**6*(x - z)/600 + delta*(delta**4*q*x**2 + delta**4*x*y - delta**4*x - delta**4*y + delta**3*x**2*y + 3*delta**3*x**2 - 4*delta**3*x*z - 3*delta**3*x - delta**3*y + 4*delta**3*z + delta**2*x**3*y + 2*delta**2*x**3 + delta**2*x**2*y - 3*delta**2*x**2*z - delta**2*x*y - 2*delta**2*x - delta**2*y + 3*delta**2*z + delta*x**4*y + delta*x**4 + 2*delta*x**3*y - 2*delta*x**3*z + delta*x**3 - 2*delta*x**2*z - delta*x**2 - 2*delta*x*y + 2*delta*x*z - delta*x - delta*y + 2*delta*z + x**5*y + 3*x**4*y - x**4*z + 2*x**3*y - 2*x**3*z - 2*x**2*y - 3*x*y + 2*x*z - y + z)/(600*varepsilon) + delta*(72*delta**3*q**2*x**3 + 108*delta**3*q*x**2*y - 108*delta**3*q*x**2 - 72*delta**3*q*x*y + 36*delta**3*x*y**2 - 72*delta**3*x*y + 36*delta**3*x - 36*delta**3*y**2 + 36*delta**3*y + 168*delta**2*q*x**3*y + 426*delta**2*q*x**3 + 24*delta**2*q*x**2*y - 486*delta**2*q*x**2*z - 216*delta**2*q*x**2 - 72*delta**2*q*x*y + 216*delta**2*q*x*z + 132*delta**2*x**2*y**2 + 222*delta**2*x**2*y - 354*delta**2*x**2 - 72*d

- \delta x + \delta z + z_{t}
+h\left(
\frac{\delta \left(q x^{2} + x y - x - y\right)}{2 \varepsilon} + \frac{\delta^{2} x - \delta^{2} z + z_{tt}}{2}
\right)
+h^2\left(
- \frac{\delta \left(\delta q x^{2} + \delta x y - \delta x - \delta y + x^{2} y - x z - y + z\right)}{6 \varepsilon} - \frac{\delta \left(2 q x + y - 1\right) \left(q x^{2} + x y - x - y\right)}{6 \varepsilon^{2}} - \frac{\delta^{3} x - \delta^{3} z - z_{ttt}}{6}
\right)
+h^3\left(
\frac{\delta \left(\delta^{2} q x^{2} + \delta^{2} x y - \delta^{2} x - \delta^{2} y + \delta x^{2} y + \delta x^{2} - 2 \delta x z - \delta x - \delta y + 2 \delta z + x^{3} y + x^{2} y - x^{2} z - x y - y + z\right)}{24 \varepsilon} + \frac{\delta \left(2 \delta q^{2} x^{3} + 3 \delta q x^{2} y - 3 \delta q x^{2} - 2 \delta q x y + \delta x y^{2} - 2 \delta x y + \delta x - \delta y^{2} + \delta y + 5 q x^{3} y + q x^{2} y - 4 q x^{2} z - 2 q x y + 2 q x z + 4 x^{2} y^{2} - 4 x^{2} y - 2 x y^{2} - 3 x y z - x y + 3 x z - 2 y^{2} + 3 y z 

In [25]:
%%time
fda = FDA(F)
for r in fda:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


0

h^2 =>


...

h^4 =>


0

h^5 =>


delta**6*(x - z)/600 + delta*(delta**4*q*x**2 + delta**4*x*y - delta**4*x - delta**4*y + delta**3*x**2*y + 3*delta**3*x**2 - 4*delta**3*x*z - 3*delta**3*x - delta**3*y + 4*delta**3*z + delta**2*x**3*y + 2*delta**2*x**3 + delta**2*x**2*y - 3*delta**2*x**2*z - delta**2*x*y - 2*delta**2*x - delta**2*y + 3*delta**2*z + delta*x**4*y + delta*x**4 + 2*delta*x**3*y - 2*delta*x**3*z + delta*x**3 - 2*delta*x**2*z - delta*x**2 - 2*delta*x*y + 2*delta*x*z - delta*x - delta*y + 2*delta*z + x**5*y + 3*x**4*y - x**4*z + 2*x**3*y - 2*x**3*z - 2*x**2*y - 3*x*y + 2*x*z - y + z)/(600*varepsilon) + delta*(72*delta**3*q**2*x**3 + 108*delta**3*q*x**2*y - 108*delta**3*q*x**2 - 72*delta**3*q*x*y + 36*delta**3*x*y**2 - 72*delta**3*x*y + 36*delta**3*x - 36*delta**3*y**2 + 36*delta**3*y + 168*delta**2*q*x**3*y + 426*delta**2*q*x**3 + 24*delta**2*q*x**2*y - 486*delta**2*q*x**2*z - 216*delta**2*q*x**2 - 72*delta**2*q*x*y + 216*delta**2*q*x*z + 132*delta**2*x**2*y**2 + 222*delta**2*x**2*y - 354*delta**2*x**2 - 72*d

- \delta x + \delta z + z_{t}
+h\left(
0
\right)
+h^2\left(
0
\right)
+h^3\left(
0
\right)
+h^4\left(
0
\right)
+h^5\left(
\frac{\delta^{6} \left(x - z\right)}{600} + \frac{\delta \left(\delta^{4} q x^{2} + \delta^{4} x y - \delta^{4} x - \delta^{4} y + \delta^{3} x^{2} y + 3 \delta^{3} x^{2} - 4 \delta^{3} x z - 3 \delta^{3} x - \delta^{3} y + 4 \delta^{3} z + \delta^{2} x^{3} y + 2 \delta^{2} x^{3} + \delta^{2} x^{2} y - 3 \delta^{2} x^{2} z - \delta^{2} x y - 2 \delta^{2} x - \delta^{2} y + 3 \delta^{2} z + \delta x^{4} y + \delta x^{4} + 2 \delta x^{3} y - 2 \delta x^{3} z + \delta x^{3} - 2 \delta x^{2} z - \delta x^{2} - 2 \delta x y + 2 \delta x z - \delta x - \delta y + 2 \delta z + x^{5} y + 3 x^{4} y - x^{4} z + 2 x^{3} y - 2 x^{3} z - 2 x^{2} y - 3 x y + 2 x z - y + z\right)}{600 \varepsilon} + \frac{\delta \left(72 \delta^{3} q^{2} x^{3} + 108 \delta^{3} q x^{2} y - 108 \delta^{3} q x^{2} - 72 \delta^{3} q x y + 36 \delta^{3} x y^{2} - 72 \delta^{3} x y + 36 \delta^{3} x - 

In [26]:
print(replace(str(fda[0][5].collect(eps, factor))))

-(y[0] - 1)*(dlt**4*y[0] - dlt**4*y[2] + dlt**3*y[0]**2 - dlt**3*y[0]*y[2] + dlt**3*y[0] - dlt**3*y[2] + dlt**2*y[0]**3 - dlt**2*y[0]**2*y[2] + 2*dlt**2*y[0]**2 - 2*dlt**2*y[0]*y[2] + dlt**2*y[0] - dlt**2*y[2] + dlt*y[0]**4 - dlt*y[0]**3*y[2] + 3*dlt*y[0]**3 - 3*dlt*y[0]**2*y[2] + 3*dlt*y[0]**2 - 3*dlt*y[0]*y[2] + dlt*y[0] - dlt*y[2] + y[0]**5*y[1] + 5*y[0]**4*y[1] - y[0]**4*y[2] + 10*y[0]**3*y[1] - 4*y[0]**3*y[2] + 10*y[0]**2*y[1] - 6*y[0]**2*y[2] + 5*y[0]*y[1] - 4*y[0]*y[2] + y[1] - y[2])/(600*eps) - (252*dlt**3*q*y[0]**3 - 216*dlt**3*q*y[0]**2*y[2] - 108*dlt**3*q*y[0]**2 + 72*dlt**3*q*y[0]*y[2] + 216*dlt**3*y[0]**2*y[1] - 216*dlt**3*y[0]**2 - 180*dlt**3*y[0]*y[1]*y[2] - 252*dlt**3*y[0]*y[1] + 180*dlt**3*y[0]*y[2] + 72*dlt**3*y[0] + 180*dlt**3*y[1]*y[2] + 36*dlt**3*y[1] - 36*dlt**3*y[2] + 372*dlt**2*q*y[0]**4 - 336*dlt**2*q*y[0]**3*y[2] + 24*dlt**2*q*y[0]**3 - 24*dlt**2*q*y[0]**2*y[2] - 108*dlt**2*q*y[0]**2 + 72*dlt**2*q*y[0]*y[2] + 852*dlt**2*y[0]**3*y[1] - 780*dlt**2*y[0]**2*y[1]*y

In [27]:
print(replace(str(fda[1][5].collect(eps, factor))))

-(dlt**5*y[0] - dlt**5*y[2] + dlt**4*y[0]**2 - dlt**4*y[0]*y[2] + dlt**4*y[0] - dlt**4*y[2] + dlt**3*y[0]**3 - dlt**3*y[0]**2*y[2] + 2*dlt**3*y[0]**2 - 2*dlt**3*y[0]*y[2] + dlt**3*y[0] - dlt**3*y[2] + dlt**2*y[0]**4 - dlt**2*y[0]**3*y[2] + 3*dlt**2*y[0]**3 - 3*dlt**2*y[0]**2*y[2] + 3*dlt**2*y[0]**2 - 3*dlt**2*y[0]*y[2] + dlt**2*y[0] - dlt**2*y[2] + dlt*y[0]**5 - dlt*y[0]**4*y[2] + 4*dlt*y[0]**4 - 4*dlt*y[0]**3*y[2] + 6*dlt*y[0]**3 - 6*dlt*y[0]**2*y[2] + 4*dlt*y[0]**2 - 4*dlt*y[0]*y[2] + dlt*y[0] - dlt*y[2] + y[0]**6*y[1] + 6*y[0]**5*y[1] - y[0]**5*y[2] + 15*y[0]**4*y[1] - 5*y[0]**4*y[2] + 20*y[0]**3*y[1] - 10*y[0]**3*y[2] + 15*y[0]**2*y[1] - 10*y[0]**2*y[2] + 6*y[0]*y[1] - 5*y[0]*y[2] + y[1] - y[2])/600 - (36*dlt**4*q*y[0]**2 + 36*dlt**4*y[0]*y[1] - 36*dlt**4*y[0] - 36*dlt**4*y[1] + 180*dlt**3*q*y[0]**3 - 144*dlt**3*q*y[0]**2*y[2] + 36*dlt**3*q*y[0]**2 + 252*dlt**3*y[0]**2*y[1] - 108*dlt**3*y[0]**2 - 180*dlt**3*y[0]*y[1]*y[2] - 180*dlt**3*y[0]*y[1] + 36*dlt**3*y[0]*y[2] - 108*dlt**3*y[

In [28]:
print(replace(str(fda[2][5].collect(eps, factor))))

dlt**6*(y[0] - y[2])/600 + dlt*(dlt**4*q*y[0]**2 + dlt**4*y[0]*y[1] - dlt**4*y[0] - dlt**4*y[1] + dlt**3*y[0]**2*y[1] + 3*dlt**3*y[0]**2 - 4*dlt**3*y[0]*y[2] - 3*dlt**3*y[0] - dlt**3*y[1] + 4*dlt**3*y[2] + dlt**2*y[0]**3*y[1] + 2*dlt**2*y[0]**3 + dlt**2*y[0]**2*y[1] - 3*dlt**2*y[0]**2*y[2] - dlt**2*y[0]*y[1] - 2*dlt**2*y[0] - dlt**2*y[1] + 3*dlt**2*y[2] + dlt*y[0]**4*y[1] + dlt*y[0]**4 + 2*dlt*y[0]**3*y[1] - 2*dlt*y[0]**3*y[2] + dlt*y[0]**3 - 2*dlt*y[0]**2*y[2] - dlt*y[0]**2 - 2*dlt*y[0]*y[1] + 2*dlt*y[0]*y[2] - dlt*y[0] - dlt*y[1] + 2*dlt*y[2] + y[0]**5*y[1] + 3*y[0]**4*y[1] - y[0]**4*y[2] + 2*y[0]**3*y[1] - 2*y[0]**3*y[2] - 2*y[0]**2*y[1] - 3*y[0]*y[1] + 2*y[0]*y[2] - y[1] + y[2])/(600*eps) + dlt*(72*dlt**3*q**2*y[0]**3 + 108*dlt**3*q*y[0]**2*y[1] - 108*dlt**3*q*y[0]**2 - 72*dlt**3*q*y[0]*y[1] + 36*dlt**3*y[0]*y[1]**2 - 72*dlt**3*y[0]*y[1] + 36*dlt**3*y[0] - 36*dlt**3*y[1]**2 + 36*dlt**3*y[1] + 168*dlt**2*q*y[0]**3*y[1] + 426*dlt**2*q*y[0]**3 + 24*dlt**2*q*y[0]**2*y[1] - 486*dlt**2*q

### Adams–Bashforth 4 order

In [29]:
%%time
F = AdamsBashforth4(f)
for r in F:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


7*x_{tt}/2 + 7*(2*q*x*x_{t} + x*y_{t} + x_{t}*y - x_{t} - y_{t})/(2*varepsilon)

h^2 =>


...

h^3 =>


-175*(delta*x_{ttt} - delta*z_{ttt} - z_{tttt})/24

h^4 =>


-887*delta*(x_{tttt} - z_{tttt})/144

- \delta x + \delta z + z_{t}
+h\left(
- \frac{7 \left(\delta x_{t} - \delta z_{t} - z_{tt}\right)}{2}
\right)
+h^2\left(
- \frac{37 \left(\delta x_{tt} - \delta z_{tt} - z_{ttt}\right)}{6}
\right)
+h^3\left(
- \frac{175 \left(\delta x_{ttt} - \delta z_{ttt} - z_{tttt}\right)}{24}
\right)
+h^4\left(
- \frac{887 \delta \left(x_{tttt} - z_{tttt}\right)}{144}
\right)

CPU times: user 1.06 s, sys: 8 ms, total: 1.06 s
Wall time: 1.06 s


In [30]:
%%time
fda = FDA(F)
for r in fda:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


0

h^2 =>


...

h^3 =>


0

h^4 =>


-887*delta**5*(x - z)/144 - 887*delta*(delta**3*q*x**2 + delta**3*x*y - delta**3*x - delta**3*y + delta**2*x**2*y + 2*delta**2*x**2 - 3*delta**2*x*z - 2*delta**2*x - delta**2*y + 3*delta**2*z + delta*x**3*y + delta*x**3 + delta*x**2*y - 2*delta*x**2*z - delta*x*y - delta*x - delta*y + 2*delta*z + x**4*y + 2*x**3*y - x**3*z - x**2*z - 2*x*y + x*z - y + z)/(144*varepsilon) - 887*delta*(2*delta**2*q**2*x**3 + 3*delta**2*q*x**2*y - 3*delta**2*q*x**2 - 2*delta**2*q*x*y + delta**2*x*y**2 - 2*delta**2*x*y + delta**2*x - delta**2*y**2 + delta**2*y + 5*delta*q*x**3*y + 6*delta*q*x**3 + delta*q*x**2*y - 9*delta*q*x**2*z - 3*delta*q*x**2 - 2*delta*q*x*y + 4*delta*q*x*z + 4*delta*x**2*y**2 + delta*x**2*y - 5*delta*x**2 - 2*delta*x*y**2 - 7*delta*x*y*z - 7*delta*x*y + 7*delta*x*z + 2*delta*x - 2*delta*y**2 + 7*delta*y*z + 2*delta*y - 2*delta*z + 8*q*x**4*y + 8*q*x**3*y - 7*q*x**3*z - 2*q*x**2*y - q*x**2*z - 2*q*x*y + 2*q*x*z + 11*x**3*y**2 - 7*x**3*y + 3*x**2*y**2 - 13*x**2*y*z - 7*x**2*y + 6*x**2*

- \delta x + \delta z + z_{t}
+h\left(
0
\right)
+h^2\left(
0
\right)
+h^3\left(
0
\right)
+h^4\left(
- \frac{887 \delta^{5} \left(x - z\right)}{144} - \frac{887 \delta \left(\delta^{3} q x^{2} + \delta^{3} x y - \delta^{3} x - \delta^{3} y + \delta^{2} x^{2} y + 2 \delta^{2} x^{2} - 3 \delta^{2} x z - 2 \delta^{2} x - \delta^{2} y + 3 \delta^{2} z + \delta x^{3} y + \delta x^{3} + \delta x^{2} y - 2 \delta x^{2} z - \delta x y - \delta x - \delta y + 2 \delta z + x^{4} y + 2 x^{3} y - x^{3} z - x^{2} z - 2 x y + x z - y + z\right)}{144 \varepsilon} - \frac{887 \delta \left(2 \delta^{2} q^{2} x^{3} + 3 \delta^{2} q x^{2} y - 3 \delta^{2} q x^{2} - 2 \delta^{2} q x y + \delta^{2} x y^{2} - 2 \delta^{2} x y + \delta^{2} x - \delta^{2} y^{2} + \delta^{2} y + 5 \delta q x^{3} y + 6 \delta q x^{3} + \delta q x^{2} y - 9 \delta q x^{2} z - 3 \delta q x^{2} - 2 \delta q x y + 4 \delta q x z + 4 \delta x^{2} y^{2} + \delta x^{2} y - 5 \delta x^{2} - 2 \delta x y^{2} - 7 \delta x y z - 7 \delta

In [31]:
print(replace(str(fda[0][4].collect(eps, factor))))

887*(y[0] - 1)*(dlt**3*y[0] - dlt**3*y[2] + dlt**2*y[0]**2 - dlt**2*y[0]*y[2] + dlt**2*y[0] - dlt**2*y[2] + dlt*y[0]**3 - dlt*y[0]**2*y[2] + 2*dlt*y[0]**2 - 2*dlt*y[0]*y[2] + dlt*y[0] - dlt*y[2] + y[0]**4*y[1] + 4*y[0]**3*y[1] - y[0]**3*y[2] + 6*y[0]**2*y[1] - 3*y[0]**2*y[2] + 4*y[0]*y[1] - 3*y[0]*y[2] + y[1] - y[2])/(144*eps) + 887*(7*dlt**2*q*y[0]**3 - 6*dlt**2*q*y[0]**2*y[2] - 3*dlt**2*q*y[0]**2 + 2*dlt**2*q*y[0]*y[2] + 6*dlt**2*y[0]**2*y[1] - 6*dlt**2*y[0]**2 - 5*dlt**2*y[0]*y[1]*y[2] - 7*dlt**2*y[0]*y[1] + 5*dlt**2*y[0]*y[2] + 2*dlt**2*y[0] + 5*dlt**2*y[1]*y[2] + dlt**2*y[1] - dlt**2*y[2] + 10*dlt*q*y[0]**4 - 9*dlt*q*y[0]**3*y[2] + dlt*q*y[0]**3 - dlt*q*y[0]**2*y[2] - 3*dlt*q*y[0]**2 + 2*dlt*q*y[0]*y[2] + 21*dlt*y[0]**3*y[1] - 9*dlt*y[0]**3 - 19*dlt*y[0]**2*y[1]*y[2] - 10*dlt*y[0]**2*y[1] - 3*dlt*y[0]**2*y[2] - dlt*y[0]**2 + 8*dlt*y[0]*y[1]*y[2] - 13*dlt*y[0]*y[1] + 10*dlt*y[0]*y[2]**2 + 13*dlt*y[0]*y[2] + 2*dlt*y[0] + 11*dlt*y[1]*y[2] + 2*dlt*y[1] - 10*dlt*y[2]**2 - 2*dlt*y[2] + 

### Adams–Bashforth 5 order

In [32]:
%%time
F = AdamsBashforth5(f)
for r in F:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


9*x_{tt}/2 + 9*(2*q*x*x_{t} + x*y_{t} + x_{t}*y - x_{t} - y_{t})/(2*varepsilon)

h^2 =>


...

h^4 =>


-2101*(delta*x_{tttt} - delta*z_{tttt} - z_{ttttt})/120

h^5 =>


-22583*delta*(x_{ttttt} - z_{ttttt})/1440

- \delta x + \delta z + z_{t}
+h\left(
- \frac{9 \left(\delta x_{t} - \delta z_{t} - z_{tt}\right)}{2}
\right)
+h^2\left(
- \frac{61 \left(\delta x_{tt} - \delta z_{tt} - z_{ttt}\right)}{6}
\right)
+h^3\left(
- \frac{123 \left(\delta x_{ttt} - \delta z_{ttt} - z_{tttt}\right)}{8}
\right)
+h^4\left(
- \frac{2101 \left(\delta x_{tttt} - \delta z_{tttt} - z_{ttttt}\right)}{120}
\right)
+h^5\left(
- \frac{22583 \delta \left(x_{ttttt} - z_{ttttt}\right)}{1440}
\right)

CPU times: user 1.54 s, sys: 16 ms, total: 1.56 s
Wall time: 1.55 s


In [33]:
%%time
fda = FDA(F)
for r in fda:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


0

h^2 =>


...

h^4 =>


0

h^5 =>


22583*delta**6*(x - z)/1440 + 22583*delta*(delta**4*q*x**2 + delta**4*x*y - delta**4*x - delta**4*y + delta**3*x**2*y + 3*delta**3*x**2 - 4*delta**3*x*z - 3*delta**3*x - delta**3*y + 4*delta**3*z + delta**2*x**3*y + 2*delta**2*x**3 + delta**2*x**2*y - 3*delta**2*x**2*z - delta**2*x*y - 2*delta**2*x - delta**2*y + 3*delta**2*z + delta*x**4*y + delta*x**4 + 2*delta*x**3*y - 2*delta*x**3*z + delta*x**3 - 2*delta*x**2*z - delta*x**2 - 2*delta*x*y + 2*delta*x*z - delta*x - delta*y + 2*delta*z + x**5*y + 3*x**4*y - x**4*z + 2*x**3*y - 2*x**3*z - 2*x**2*y - 3*x*y + 2*x*z - y + z)/(1440*varepsilon) + 22583*delta*(2*delta**3*q**2*x**3 + 3*delta**3*q*x**2*y - 3*delta**3*q*x**2 - 2*delta**3*q*x*y + delta**3*x*y**2 - 2*delta**3*x*y + delta**3*x - delta**3*y**2 + delta**3*y + 5*delta**2*q*x**3*y + 13*delta**2*q*x**3 + delta**2*q*x**2*y - 15*delta**2*q*x**2*z - 6*delta**2*q*x**2 - 2*delta**2*q*x*y + 6*delta**2*q*x*z + 4*delta**2*x**2*y**2 + 7*delta**2*x**2*y - 11*delta**2*x**2 - 2*delta**2*x*y**2 - 

- \delta x + \delta z + z_{t}
+h\left(
0
\right)
+h^2\left(
0
\right)
+h^3\left(
0
\right)
+h^4\left(
0
\right)
+h^5\left(
\frac{22583 \delta^{6} \left(x - z\right)}{1440} + \frac{22583 \delta \left(\delta^{4} q x^{2} + \delta^{4} x y - \delta^{4} x - \delta^{4} y + \delta^{3} x^{2} y + 3 \delta^{3} x^{2} - 4 \delta^{3} x z - 3 \delta^{3} x - \delta^{3} y + 4 \delta^{3} z + \delta^{2} x^{3} y + 2 \delta^{2} x^{3} + \delta^{2} x^{2} y - 3 \delta^{2} x^{2} z - \delta^{2} x y - 2 \delta^{2} x - \delta^{2} y + 3 \delta^{2} z + \delta x^{4} y + \delta x^{4} + 2 \delta x^{3} y - 2 \delta x^{3} z + \delta x^{3} - 2 \delta x^{2} z - \delta x^{2} - 2 \delta x y + 2 \delta x z - \delta x - \delta y + 2 \delta z + x^{5} y + 3 x^{4} y - x^{4} z + 2 x^{3} y - 2 x^{3} z - 2 x^{2} y - 3 x y + 2 x z - y + z\right)}{1440 \varepsilon} + \frac{22583 \delta \left(2 \delta^{3} q^{2} x^{3} + 3 \delta^{3} q x^{2} y - 3 \delta^{3} q x^{2} - 2 \delta^{3} q x y + \delta^{3} x y^{2} - 2 \delta^{3} x y + \delta^{

In [34]:
print(replace(str(fda[0][5].collect(eps, factor))))

-22583*(y[0] - 1)*(dlt**4*y[0] - dlt**4*y[2] + dlt**3*y[0]**2 - dlt**3*y[0]*y[2] + dlt**3*y[0] - dlt**3*y[2] + dlt**2*y[0]**3 - dlt**2*y[0]**2*y[2] + 2*dlt**2*y[0]**2 - 2*dlt**2*y[0]*y[2] + dlt**2*y[0] - dlt**2*y[2] + dlt*y[0]**4 - dlt*y[0]**3*y[2] + 3*dlt*y[0]**3 - 3*dlt*y[0]**2*y[2] + 3*dlt*y[0]**2 - 3*dlt*y[0]*y[2] + dlt*y[0] - dlt*y[2] + y[0]**5*y[1] + 5*y[0]**4*y[1] - y[0]**4*y[2] + 10*y[0]**3*y[1] - 4*y[0]**3*y[2] + 10*y[0]**2*y[1] - 6*y[0]**2*y[2] + 5*y[0]*y[1] - 4*y[0]*y[2] + y[1] - y[2])/(1440*eps) - 22583*(8*dlt**3*q*y[0]**3 - 7*dlt**3*q*y[0]**2*y[2] - 3*dlt**3*q*y[0]**2 + 2*dlt**3*q*y[0]*y[2] + 7*dlt**3*y[0]**2*y[1] - 7*dlt**3*y[0]**2 - 6*dlt**3*y[0]*y[1]*y[2] - 8*dlt**3*y[0]*y[1] + 6*dlt**3*y[0]*y[2] + 2*dlt**3*y[0] + 6*dlt**3*y[1]*y[2] + dlt**3*y[1] - dlt**3*y[2] + 12*dlt**2*q*y[0]**4 - 11*dlt**2*q*y[0]**3*y[2] + dlt**2*q*y[0]**3 - dlt**2*q*y[0]**2*y[2] - 3*dlt**2*q*y[0]**2 + 2*dlt**2*q*y[0]*y[2] + 28*dlt**2*y[0]**3*y[1] - 26*dlt**2*y[0]**2*y[1]*y[2] - 12*dlt**2*y[0]**2*y[

### Adams–Moulton 4 order

In [35]:
%%time
F = AdamsMoulton4(f)
for r in F:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


7*x_{tt}/2 + 7*(2*q*x*x_{t} + x*y_{t} + x_{t}*y - x_{t} - y_{t})/(2*varepsilon)

h^2 =>


...

h^3 =>


-175*(delta*x_{ttt} - delta*z_{ttt} - z_{tttt})/24

h^4 =>


-781*delta*(x_{tttt} - z_{tttt})/120

- \delta x + \delta z + z_{t}
+h\left(
- \frac{7 \left(\delta x_{t} - \delta z_{t} - z_{tt}\right)}{2}
\right)
+h^2\left(
- \frac{37 \left(\delta x_{tt} - \delta z_{tt} - z_{ttt}\right)}{6}
\right)
+h^3\left(
- \frac{175 \left(\delta x_{ttt} - \delta z_{ttt} - z_{tttt}\right)}{24}
\right)
+h^4\left(
- \frac{781 \delta \left(x_{tttt} - z_{tttt}\right)}{120}
\right)

CPU times: user 1e+03 ms, sys: 8 ms, total: 1.01 s
Wall time: 1 s


In [36]:
%%time
fda = FDA(F)
for r in fda:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


0

h^2 =>


...

h^3 =>


0

h^4 =>


-781*delta**5*(x - z)/120 - 781*delta*(delta**3*q*x**2 + delta**3*x*y - delta**3*x - delta**3*y + delta**2*x**2*y + 2*delta**2*x**2 - 3*delta**2*x*z - 2*delta**2*x - delta**2*y + 3*delta**2*z + delta*x**3*y + delta*x**3 + delta*x**2*y - 2*delta*x**2*z - delta*x*y - delta*x - delta*y + 2*delta*z + x**4*y + 2*x**3*y - x**3*z - x**2*z - 2*x*y + x*z - y + z)/(120*varepsilon) - 781*delta*(2*delta**2*q**2*x**3 + 3*delta**2*q*x**2*y - 3*delta**2*q*x**2 - 2*delta**2*q*x*y + delta**2*x*y**2 - 2*delta**2*x*y + delta**2*x - delta**2*y**2 + delta**2*y + 5*delta*q*x**3*y + 6*delta*q*x**3 + delta*q*x**2*y - 9*delta*q*x**2*z - 3*delta*q*x**2 - 2*delta*q*x*y + 4*delta*q*x*z + 4*delta*x**2*y**2 + delta*x**2*y - 5*delta*x**2 - 2*delta*x*y**2 - 7*delta*x*y*z - 7*delta*x*y + 7*delta*x*z + 2*delta*x - 2*delta*y**2 + 7*delta*y*z + 2*delta*y - 2*delta*z + 8*q*x**4*y + 8*q*x**3*y - 7*q*x**3*z - 2*q*x**2*y - q*x**2*z - 2*q*x*y + 2*q*x*z + 11*x**3*y**2 - 7*x**3*y + 3*x**2*y**2 - 13*x**2*y*z - 7*x**2*y + 6*x**2*

- \delta x + \delta z + z_{t}
+h\left(
0
\right)
+h^2\left(
0
\right)
+h^3\left(
0
\right)
+h^4\left(
- \frac{781 \delta^{5} \left(x - z\right)}{120} - \frac{781 \delta \left(\delta^{3} q x^{2} + \delta^{3} x y - \delta^{3} x - \delta^{3} y + \delta^{2} x^{2} y + 2 \delta^{2} x^{2} - 3 \delta^{2} x z - 2 \delta^{2} x - \delta^{2} y + 3 \delta^{2} z + \delta x^{3} y + \delta x^{3} + \delta x^{2} y - 2 \delta x^{2} z - \delta x y - \delta x - \delta y + 2 \delta z + x^{4} y + 2 x^{3} y - x^{3} z - x^{2} z - 2 x y + x z - y + z\right)}{120 \varepsilon} - \frac{781 \delta \left(2 \delta^{2} q^{2} x^{3} + 3 \delta^{2} q x^{2} y - 3 \delta^{2} q x^{2} - 2 \delta^{2} q x y + \delta^{2} x y^{2} - 2 \delta^{2} x y + \delta^{2} x - \delta^{2} y^{2} + \delta^{2} y + 5 \delta q x^{3} y + 6 \delta q x^{3} + \delta q x^{2} y - 9 \delta q x^{2} z - 3 \delta q x^{2} - 2 \delta q x y + 4 \delta q x z + 4 \delta x^{2} y^{2} + \delta x^{2} y - 5 \delta x^{2} - 2 \delta x y^{2} - 7 \delta x y z - 7 \delta

In [37]:
print(replace(str(fda[0][4].collect(eps, factor))))

781*(y[0] - 1)*(dlt**3*y[0] - dlt**3*y[2] + dlt**2*y[0]**2 - dlt**2*y[0]*y[2] + dlt**2*y[0] - dlt**2*y[2] + dlt*y[0]**3 - dlt*y[0]**2*y[2] + 2*dlt*y[0]**2 - 2*dlt*y[0]*y[2] + dlt*y[0] - dlt*y[2] + y[0]**4*y[1] + 4*y[0]**3*y[1] - y[0]**3*y[2] + 6*y[0]**2*y[1] - 3*y[0]**2*y[2] + 4*y[0]*y[1] - 3*y[0]*y[2] + y[1] - y[2])/(120*eps) + 781*(7*dlt**2*q*y[0]**3 - 6*dlt**2*q*y[0]**2*y[2] - 3*dlt**2*q*y[0]**2 + 2*dlt**2*q*y[0]*y[2] + 6*dlt**2*y[0]**2*y[1] - 6*dlt**2*y[0]**2 - 5*dlt**2*y[0]*y[1]*y[2] - 7*dlt**2*y[0]*y[1] + 5*dlt**2*y[0]*y[2] + 2*dlt**2*y[0] + 5*dlt**2*y[1]*y[2] + dlt**2*y[1] - dlt**2*y[2] + 10*dlt*q*y[0]**4 - 9*dlt*q*y[0]**3*y[2] + dlt*q*y[0]**3 - dlt*q*y[0]**2*y[2] - 3*dlt*q*y[0]**2 + 2*dlt*q*y[0]*y[2] + 21*dlt*y[0]**3*y[1] - 9*dlt*y[0]**3 - 19*dlt*y[0]**2*y[1]*y[2] - 10*dlt*y[0]**2*y[1] - 3*dlt*y[0]**2*y[2] - dlt*y[0]**2 + 8*dlt*y[0]*y[1]*y[2] - 13*dlt*y[0]*y[1] + 10*dlt*y[0]*y[2]**2 + 13*dlt*y[0]*y[2] + 2*dlt*y[0] + 11*dlt*y[1]*y[2] + 2*dlt*y[1] - 10*dlt*y[2]**2 - 2*dlt*y[2] + 

### Adams–Moulton 5 order

In [38]:
%%time
F = AdamsMoulton5(f)
for r in F:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


9*x_{tt}/2 + 9*(2*q*x*x_{t} + x*y_{t} + x_{t}*y - x_{t} - y_{t})/(2*varepsilon)

h^2 =>


...

h^4 =>


-2101*(delta*x_{tttt} - delta*z_{tttt} - z_{ttttt})/120

h^5 =>


-1281*delta*(x_{ttttt} - z_{ttttt})/80

- \delta x + \delta z + z_{t}
+h\left(
- \frac{9 \left(\delta x_{t} - \delta z_{t} - z_{tt}\right)}{2}
\right)
+h^2\left(
- \frac{61 \left(\delta x_{tt} - \delta z_{tt} - z_{ttt}\right)}{6}
\right)
+h^3\left(
- \frac{123 \left(\delta x_{ttt} - \delta z_{ttt} - z_{tttt}\right)}{8}
\right)
+h^4\left(
- \frac{2101 \left(\delta x_{tttt} - \delta z_{tttt} - z_{ttttt}\right)}{120}
\right)
+h^5\left(
- \frac{1281 \delta \left(x_{ttttt} - z_{ttttt}\right)}{80}
\right)

CPU times: user 1.56 s, sys: 12 ms, total: 1.57 s
Wall time: 1.57 s


In [39]:
fda = FDA(F)
for r in fda:
    r.prn(eps)
    r.prnlatex(eps)
    print()

h^0 =>


x_{t} + (q*x**2 + x*y - x - y)/varepsilon

h^1 =>


0

h^2 =>


...

h^4 =>


0

h^5 =>


1281*delta**6*(x - z)/80 + 1281*delta*(delta**4*q*x**2 + delta**4*x*y - delta**4*x - delta**4*y + delta**3*x**2*y + 3*delta**3*x**2 - 4*delta**3*x*z - 3*delta**3*x - delta**3*y + 4*delta**3*z + delta**2*x**3*y + 2*delta**2*x**3 + delta**2*x**2*y - 3*delta**2*x**2*z - delta**2*x*y - 2*delta**2*x - delta**2*y + 3*delta**2*z + delta*x**4*y + delta*x**4 + 2*delta*x**3*y - 2*delta*x**3*z + delta*x**3 - 2*delta*x**2*z - delta*x**2 - 2*delta*x*y + 2*delta*x*z - delta*x - delta*y + 2*delta*z + x**5*y + 3*x**4*y - x**4*z + 2*x**3*y - 2*x**3*z - 2*x**2*y - 3*x*y + 2*x*z - y + z)/(80*varepsilon) + 1281*delta*(2*delta**3*q**2*x**3 + 3*delta**3*q*x**2*y - 3*delta**3*q*x**2 - 2*delta**3*q*x*y + delta**3*x*y**2 - 2*delta**3*x*y + delta**3*x - delta**3*y**2 + delta**3*y + 5*delta**2*q*x**3*y + 13*delta**2*q*x**3 + delta**2*q*x**2*y - 15*delta**2*q*x**2*z - 6*delta**2*q*x**2 - 2*delta**2*q*x*y + 6*delta**2*q*x*z + 4*delta**2*x**2*y**2 + 7*delta**2*x**2*y - 11*delta**2*x**2 - 2*delta**2*x*y**2 - 12*delt

- \delta x + \delta z + z_{t}
+h\left(
0
\right)
+h^2\left(
0
\right)
+h^3\left(
0
\right)
+h^4\left(
0
\right)
+h^5\left(
\frac{1281 \delta^{6} \left(x - z\right)}{80} + \frac{1281 \delta \left(\delta^{4} q x^{2} + \delta^{4} x y - \delta^{4} x - \delta^{4} y + \delta^{3} x^{2} y + 3 \delta^{3} x^{2} - 4 \delta^{3} x z - 3 \delta^{3} x - \delta^{3} y + 4 \delta^{3} z + \delta^{2} x^{3} y + 2 \delta^{2} x^{3} + \delta^{2} x^{2} y - 3 \delta^{2} x^{2} z - \delta^{2} x y - 2 \delta^{2} x - \delta^{2} y + 3 \delta^{2} z + \delta x^{4} y + \delta x^{4} + 2 \delta x^{3} y - 2 \delta x^{3} z + \delta x^{3} - 2 \delta x^{2} z - \delta x^{2} - 2 \delta x y + 2 \delta x z - \delta x - \delta y + 2 \delta z + x^{5} y + 3 x^{4} y - x^{4} z + 2 x^{3} y - 2 x^{3} z - 2 x^{2} y - 3 x y + 2 x z - y + z\right)}{80 \varepsilon} + \frac{1281 \delta \left(2 \delta^{3} q^{2} x^{3} + 3 \delta^{3} q x^{2} y - 3 \delta^{3} q x^{2} - 2 \delta^{3} q x y + \delta^{3} x y^{2} - 2 \delta^{3} x y + \delta^{3} x - 

In [40]:
print(replace(str(fda[0][5].collect(eps, factor))))

-1281*(y[0] - 1)*(dlt**4*y[0] - dlt**4*y[2] + dlt**3*y[0]**2 - dlt**3*y[0]*y[2] + dlt**3*y[0] - dlt**3*y[2] + dlt**2*y[0]**3 - dlt**2*y[0]**2*y[2] + 2*dlt**2*y[0]**2 - 2*dlt**2*y[0]*y[2] + dlt**2*y[0] - dlt**2*y[2] + dlt*y[0]**4 - dlt*y[0]**3*y[2] + 3*dlt*y[0]**3 - 3*dlt*y[0]**2*y[2] + 3*dlt*y[0]**2 - 3*dlt*y[0]*y[2] + dlt*y[0] - dlt*y[2] + y[0]**5*y[1] + 5*y[0]**4*y[1] - y[0]**4*y[2] + 10*y[0]**3*y[1] - 4*y[0]**3*y[2] + 10*y[0]**2*y[1] - 6*y[0]**2*y[2] + 5*y[0]*y[1] - 4*y[0]*y[2] + y[1] - y[2])/(80*eps) - 1281*(8*dlt**3*q*y[0]**3 - 7*dlt**3*q*y[0]**2*y[2] - 3*dlt**3*q*y[0]**2 + 2*dlt**3*q*y[0]*y[2] + 7*dlt**3*y[0]**2*y[1] - 7*dlt**3*y[0]**2 - 6*dlt**3*y[0]*y[1]*y[2] - 8*dlt**3*y[0]*y[1] + 6*dlt**3*y[0]*y[2] + 2*dlt**3*y[0] + 6*dlt**3*y[1]*y[2] + dlt**3*y[1] - dlt**3*y[2] + 12*dlt**2*q*y[0]**4 - 11*dlt**2*q*y[0]**3*y[2] + dlt**2*q*y[0]**3 - dlt**2*q*y[0]**2*y[2] - 3*dlt**2*q*y[0]**2 + 2*dlt**2*q*y[0]*y[2] + 28*dlt**2*y[0]**3*y[1] - 26*dlt**2*y[0]**2*y[1]*y[2] - 12*dlt**2*y[0]**2*y[1] -